# 黑马头条项目｜第10天：定时计算热点文章与 XXL-JOB

> 学习目标：理解分布式任务调度，使用 XXL-JOB 定时计算热点文章分值，把每个频道的 Top 30 缓存到 Redis，并改造 APP 首页文章查询优先读取热点缓存。

本篇依据百度网盘 `day10-定时计算热点文章、xxl-job/讲义/热点文章-定时计算.md` 的网页预览整理。讲义涵盖 XXL-JOB 概述、调度中心与执行器、Cron、阻塞处理、路由策略、分片广播、热点文章计算和首页接口改造；未读取代码目录、PPT、视频、资料目录或数据库脚本，未启动 MySQL、XXL-JOB、Redis、Nacos、Feign、微服务或真实定时任务。

课程示例中的内网地址、默认账号、密码和邮件配置均用占位符表示。文档中展示的代码有教学性质，下面会区分“讲义事实”“源码示意”和“工程补充”。

资料：[黑马头条项目分享链接](https://pan.baidu.com/s/1u7PNExeoEsYGh5pAnSyocg?pwd=9987)。


## 1. 为什么要计算热点文章

现有文章列表按发布时间倒序查询，会出现两个问题：访问量较大时请求直接压到数据库；新发布文章排在前面，但它未必是用户真正关心的热点。

第10天的目标是把文章热度结果放入 Redis。APP 查询文章列表时，首页优先读取热度较高的文章，其他场景继续使用原有列表查询。

~~~text
文章行为数据（阅读、点赞、评论、收藏）
                 ↓ 定时计算
              热度分值
                 ↓ 排序、取Top 30
Redis：每个频道一份热点列表 + 全部推荐列表
                 ↓
APP首页文章查询优先读取Redis，缓存无数据时回退数据库
~~~

热点缓存是文章查询的读模型，不取代 `ap_article` 主表，也不等于搜索索引。文章上下架、删除、行为变化和定时重算之间需要明确最终一致性边界。


## 2. 热点分值与数据来源

讲义定义的热度标准是四类行为：**阅读、点赞、评论、收藏**。示例常量给出点赞权重3、评论权重5、收藏权重8，阅读没有单独常量，示例中按1计分：

~~~text
score = views × 1 + likes × 3 + comment × 5 + collection × 8
~~~

字段来自 `ApArticle` 的文章统计属性，示例代码对空值按0处理。`HotArticleVo` 继承 `ApArticle`，新增 `Integer score`，便于把原文章字段和计算分值一起缓存。

| 行为 | 示例权重 | 课程常量/含义 |
|---|---:|---|
| 阅读 `views` | 1 | 示例直接累加 |
| 点赞 `likes` | 3 | `HOT_ARTICLE_LIKE_WEIGHT` |
| 评论 `comment` | 5 | `HOT_ARTICLE_COMMENT_WEIGHT` |
| 收藏 `collection` | 8 | `HOT_ARTICLE_COLLECTION_WEIGHT` |

讲义没有时间衰减、去重、异常刷量或分值溢出规则。真实系统至少要定义统计窗口、计数口径、权重配置来源、`Integer` 溢出保护和同分排序规则。热点分值是产品策略，不应把示例权重当作普适算法。


## 3. 定时计算的业务流程

讲义的实现步骤是：查询最近一段时间内的有效文章；为每篇文章计算分值，组装 `HotArticleVo`；查询所有频道；逐频道筛选文章，按分值倒序取30条写 Redis；再把全部文章取30条写入默认推荐 key；APP 首页首屏优先读 Redis，缓存没有内容再走原有 `load` 查询。

~~~text
XXL-JOB调度中心
       ↓ HTTP调度
article执行器 ComputeHotArticleJob
       ↓
HotArticleService.computeHotArticle()
       ├── 查询有效文章
       ├── 计算HotArticleVo.score
       ├── Feign查询频道
       └── Redis写入频道key和默认key
                       ↓
APP /article/load → 首页？ → Redis热点列表 / 数据库回退
~~~

该任务不需要前端 API，讲义明确称其为纯后台功能；但缓存读取会改变已有列表接口的实现路径。


## 4. 什么是分布式任务调度

微服务会部署多个实例，跨多个实例执行定时任务就属于分布式任务调度。它带来并行调度、高可用、弹性扩容以及统一管理与监测。

集群也带来“同一个任务执行多次”的问题。热点任务在多个 article 实例上同时执行，会重复查询、排序和写缓存；有副作用的任务还会造成重复业务操作。

常见控制方式：

1. **分布式锁**：执行前竞争锁，未拿到锁的实例跳过；
2. **Leader 选举**：只有 Leader 节点执行；
3. **调度中心单次派发**：由中心选择一个执行器，其他节点不重复接收。

XXL-JOB通过调度中心、执行器注册和路由策略完成任务派发；具体任务仍要保证幂等，不能把“调度中心只派一次”当作业务绝对保证。


## 5. XXL-JOB 的角色与能力

讲义将 XXL-JOB 定义为轻量级分布式任务调度平台，目标是开发迅速、学习简单、轻量、易扩展，并提供 Web 管理页面。

| 角色 | 作用 |
|---|---|
| 调度中心 `xxl-job-admin` | 保存任务配置、触发调度、选择执行器、管理日志和告警 |
| 执行器 | 集成到业务微服务，注册到调度中心并执行 JobHandler |
| 任务 | 调度中心中的配置，绑定一个执行器和一个 JobHandler |
| 调度数据库 | 保存锁、执行器、任务、日志、注册表和用户等信息 |

~~~text
管理员配置 Cron/执行器/JobHandler
               ↓
调度中心选择在线执行器并发起调度
               ↓
业务执行器找到 @XxlJob 方法
               ↓
任务成功/失败回调，调度中心记录日志和告警
~~~

讲义列举的能力包括任务 CRUD、Cron、固定速度、依赖任务、路由策略、失败重试、阻塞策略、执行日志和邮件/扩展告警。实际可用能力受版本和部署配置影响。


## 6. 调度中心数据库与部署边界

讲义要求使用 XXL-JOB 源码中的 `/xxl-job/doc/db/tables_xxl_job.sql` 初始化调度数据库，共8张表，预览中列出：

- `xxl_job_lock`：任务调度锁；
- `xxl_job_group`：执行器信息；
- `xxl_job_info`：任务扩展配置，如分组、Cron、执行器、参数和告警邮箱；
- `xxl_job_log`：调度和执行历史；
- `xxl_job_logglue`：GLUE版本历史；
- `xxl_job_registry`：执行器与调度中心注册信息；
- `xxl_job_user`：调度中心用户。

讲义文字称“共8张表”，可见列举项为7个，剩余表名未在预览文本中完整显示；不能凭空补写表名。调度中心集群节点要连接同一个 MySQL；如果数据库主从，调度中心应强制走主库，以免锁和任务配置读到延迟副本。

讲义提供源码启动、Docker部署两种方式。本篇只记录结构，不执行 Docker、MySQL 初始化或服务启动。实际部署应将数据库凭据、调度 access token、邮件配置放入安全配置，并保护管理页面。


## 7. 调度中心任务配置

### 基础配置

- 执行器：任务必须绑定的执行器分组；
- 任务描述：方便管理和检索；
- 负责人：问题定位和责任归属；
- 报警邮件：调度失败时通知的邮箱列表。

### 调度配置

- **无**：不主动触发；
- **CRON**：按 Cron 表达式触发；
- **固定速度**：按固定间隔周期触发。

### 任务配置

- **BEAN 模式**：在执行器端维护 JobHandler；
- **JobHandler**：匹配执行器中 `@XxlJob` 的值；
- **执行参数**：传给任务的业务参数。

讲义示例使用每天凌晨2点的 Cron `0 0 2 * * ?`。它是课程任务的示例约定；生产应根据数据窗口、数据库峰值和业务时区配置，并为修改保留审核、版本和回滚记录。


## 8. 执行器接入与 JobHandler

讲义的接入步骤：引入 `com.xuxueli:xxl-job-core:2.3.0`；配置调度中心地址、执行器 `appname` 和端口；创建 `XxlJobSpringExecutor` Bean；在 Spring Bean 方法上添加 `@XxlJob("JobHandler")`；在调度中心创建任务并匹配同名 JobHandler。

配置结构示意（地址、端口使用占位符）：

~~~yaml
xxl:
  job:
    admin:
      addresses: <XXL_JOB_ADMIN_URL>
    executor:
      appname: <EXECUTOR_APP_NAME>
      port: <EXECUTOR_PORT>
~~~

代码示意：

~~~java
@Component
public class ExampleJob {
    @XxlJob("exampleJobHandler")
    public void execute() {
        // 校验状态、执行幂等更新、记录结果
    }
}
~~~

`appname` 是执行器集群的唯一标识，调度中心以它发现在线节点。执行器端口要避免多个实例冲突，并在容器、网关、防火墙和健康检查中开放正确范围。


## 9. 路由策略与阻塞处理

讲义列出 FIRST、LAST、ROUND、RANDOM、CONSISTENT_HASH、LEAST_FREQUENTLY_USED、LEAST_RECENTLY_USED、FAILOVER、BUSYOVER、SHARDING_BROADCAST。

| 策略 | 适用理解 |
|---|---|
| ROUND | 轮询节点，分摊任务 |
| RANDOM | 随机选择在线节点 |
| FAILOVER | 按顺序心跳检测，选择可用节点 |
| BUSYOVER | 按顺序检查空闲状态，选择空闲节点 |
| SHARDING_BROADCAST | 所有节点都执行，并收到分片参数 |

热点任务的课程配置使用轮询。轮询不是“所有节点同时执行”，一次调度只选择一个在线执行器。

阻塞处理策略有：

- **单机串行**：后续调度进入 FIFO 队列；
- **丢弃后续调度**：当前执行器忙时，本次请求丢弃并标记失败；
- **覆盖之前调度**：终止运行中的任务、清空队列，运行新任务。

覆盖策略对不可中断或有外部副作用的任务有风险。选择策略应结合任务耗时、是否可重入、是否允许漏算和是否有补偿。


## 10. 分片广播与并行任务

分片广播时，一次调度会广播给执行器集群的所有节点，每个节点拿到：

- `index`：当前分片序号，从0开始；
- `total`：分片总数，即执行器集群机器数。

讲义用10000个任务、两个节点举例，让每个节点处理约5000个。示意分片规则：

~~~java
int shardIndex = XxlJobHelper.getShardIndex();
int shardTotal = XxlJobHelper.getShardTotal();
for (Integer item : items) {
    if (item % shardTotal == shardIndex) {
        process(item);
    }
}
~~~

这种取模分片要求任务集合可枚举、分片总数稳定、处理逻辑幂等。节点数量变化会改变分配结果，失败重试要依靠断点、幂等键或任务状态记录。

分片广播适合可拆分批处理；热点文章课程实现是先查询文章再按频道分组，并没有在热点任务代码中使用分片广播，不要把两个示例混写。


## 11. 热点文章业务规则

课程需求：**为每个频道缓存热度较高的30条文章优先展示**。

缓存维度：

| 维度 | Redis key示意 | 内容 |
|---|---|---|
| 频道 | `hot_article_first_page_<channelId>` | 该频道按分值倒序的30条 `HotArticleVo` |
| 全部推荐 | `hot_article_first_page___all__` | 全部文章按分值倒序的30条 |

讲义常量：

~~~java
HOT_ARTICLE_LIKE_WEIGHT = 3;
HOT_ARTICLE_COMMENT_WEIGHT = 5;
HOT_ARTICLE_COLLECTION_WEIGHT = 8;
HOT_ARTICLE_FIRST_PAGE = "hot_article_first_page_";
DEFAULT_TAG = "__all__";
~~~

文章先按分值降序排序，数量超过30时截取前30条；少于30条则全部缓存。频道文章按 `channelId` 匹配，默认推荐列表不做频道过滤。

工程补充：增加发布时间、文章ID等次排序字段，避免同分文章抖动；缓存只放首页所需字段，避免暴露内部或隐私字段。


## 12. 查询有效文章与时间窗口

讲义要求在 `ApArticleMapper.xml` 增加 `findArticleListByLast5days`，查询 `ap_article` 与 `ap_article_config`：

~~~sql
SELECT aa.*
FROM ap_article aa
LEFT JOIN ap_article_config aac ON aa.id = aac.article_id
WHERE aac.is_delete != 1
  AND aac.is_down != 1
  AND aa.publish_time >= #{dayParam}
~~~

业务意图是排除删除和下架文章，并限定发布时间窗口。`LEFT JOIN` 加右表 WHERE 条件后，效果接近要求存在有效配置的内连接；配置缺失是否应展示，需要数据库约束或产品规则明确。

**讲义存在明显不一致：** 方法名为 `findArticleListByLast5days`，文字写“查询前5天”，但展示的实现代码使用 `DateTime.now().minusDays(50)`。这应作为资料冲突记录，不能擅自认定50天正确。实现前要统一需求窗口、方法名、注释、测试数据和监控。

还要明确时区、未来发布时间、`publish_time IS NOT NULL`、频道为空以及查询索引。文章量增长后，应采用分页、分片、数据库排序/聚合或时间窗口批处理，避免一次性加载全量数据。


## 13. 热点计算服务结构

讲义定义：

~~~java
public interface HotArticleService {
    void computeHotArticle();
}
~~~

实现逻辑分三步：

1. `apArticleMapper.findArticleListByLast5days(dayParam)` 查询文章；
2. `computeHotArticle(apArticleList)` 复制为 `HotArticleVo` 并计算 score；
3. `cacheTagToRedis(hotArticleVoList)` 查询频道、按频道排序缓存，同时缓存默认推荐。

~~~text
computeHotArticle()
    ├── 查询文章
    ├── 每篇：views + likes×3 + comment×5 + collection×8
    ├── Feign：获取频道列表
    ├── 每个频道：过滤channelId → score倒序 → 取30 → Redis
    └── DEFAULT_TAG：全部文章 → score倒序 → 取30 → Redis
~~~

讲义示例使用 `BeanUtils.copyProperties`、Java Stream、FastJSON 和 `CacheService`，并给计算方法加 `@Transactional`。本地事务不能覆盖远程 Feign 和 Redis 写入；任务失败可能已写入部分频道，需要批次版本、失败重试或保留旧缓存。


## 14. 频道远程接口与 Feign

为每个频道缓存数据，article服务需要查询所有频道。讲义要求：在 `heima-leadnews-feign-api` 定义 `IWemediaClient`；使用 `@FeignClient("leadnews-wemedia")`；通过 `GET /api/v1/channel/list` 调用 wemedia；wemedia端调用 `WmChannelService.findAll()`；在 `ArticleApplication` 增加 `@EnableFeignClients(basePackages = "com.heima.apis")`。

接口示意：

~~~java
@FeignClient("leadnews-wemedia")
public interface IWemediaClient {
    @GetMapping("/api/v1/channel/list")
    ResponseResult getChannels();
}
~~~

响应转换为 `List<WmChannel>` 后，用 `wmChannel.getId()` 与文章 `channelId` 过滤。响应码必须检查；频道服务不可用时不能把空列表误当成“没有频道”，否则可能只更新默认推荐或清空频道缓存。

工程补充：设置连接/读取超时、重试边界、熔断和 requestId；频道列表可短时缓存。频道配置与热点任务并发时，要决定使用任务开始时快照还是最新列表。


## 15. Redis 写入与缓存发布

课程通过 `CacheService.set(key, JSON.toJSONString(hotArticleVos))` 写入 JSON 数组。单 key 的直接流程是：筛选并排序 → 取30 → 序列化 → SET频道key。

逐频道写入存在半批风险：读请求可能看到一部分频道的新数据、另一部分旧数据；某个频道写成功后任务失败，也会保留半更新状态。

工程补充的可靠发布方向：

- 使用批次版本（如 `hot:v<batch>:<tag>`），全部写完后再切换当前版本指针；
- 记录计算窗口、最后成功时间和缓存 schema 版本；
- 写入失败时保留上一批有效缓存，不立即删除旧 key；
- 默认推荐和频道列表使用同一批次边界；
- JSON解析失败时回退数据库并告警。

这些属于可靠缓存发布方案，讲义只展示直接 `set`。缓存缺失时，服务应回退原数据库查询，而不是返回空白首页。


## 16. XXL-JOB 热点任务接入

讲义的接入步骤：

1. article服务 pom 引入 `xxl-job-core` 2.3.0；
2. 调度中心新建执行器 `leadnews-hot-article-executor`；
3. 新建任务，路由策略选择轮询，Cron示例为 `0 0 2 * * ?`；
4. article服务配置 `XxlJobSpringExecutor`；
5. Nacos中配置调度中心地址、执行器 appname 和端口；
6. 创建 `ComputeHotArticleJob`，用 `@XxlJob("computeHotArticleJob")` 调用 `hotArticleService.computeHotArticle()`。

任务示意：

~~~java
@Component
public class ComputeHotArticleJob {
    @Autowired
    private HotArticleService hotArticleService;

    @XxlJob("computeHotArticleJob")
    public void handle() {
        hotArticleService.computeHotArticle();
    }
}
~~~

任务日志应包含批次、窗口、文章数、频道数、成功/失败数量和耗时，但不要记录数据库密码、access token或完整文章正文。下游失败应抛出可识别异常，让调度中心执行重试和告警，而不是只打印日志后伪装成功。


## 17. 首页文章查询改造

讲义在 `ApArticleService` 新增：

~~~java
ResponseResult load2(ArticleHomeDto dto, Short type, boolean firstPage);
~~~

服务逻辑：

~~~text
firstPage=true?
    ├─ 是：GET hot_article_first_page_<dto.tag>
    │       ├─ 有缓存 → JSON解析HotArticleVo列表并返回
    │       └─ 无缓存 → 调用原load(type,dto)
    └─ 否：调用原load(type,dto)
~~~

控制器把首页 `/load` 改为调用：

~~~java
apArticleService.load2(dto, ArticleConstants.LOADTYPE_LOAD_MORE, true);
~~~

注意：`firstPage`只代表首页首屏，加载更多不应每次读热点首屏缓存；`dto.getTag()`必须和频道ID/`__all__`约定一致；缓存字段要兼容原首页响应。缓存文章可能已下架，必要时在读取侧做可见性校验。


## 18. 首屏缓存的正确性边界

定时缓存与实时内容存在时间差：

~~~text
02:00 任务计算并写入热点
02:10 新文章发布/下架/用户大量点赞
02:20 首页读取仍可能看到02:00快照
~~~

定时计算是用可接受的新鲜度换取查询性能。产品需要明确热点结果允许延迟多久、下架是否必须立即消失、紧急内容是否能置顶或剔除。

可组合的改进：上下架事件主动删除/过滤缓存中的文章；新文章等待下一次计算；行为事件更新统计源而不是每次重排全量频道；任务失败保留上一批并告警；查询时做轻量可见性校验。

不要把“缓存命中”理解为“数据库状态绝对最新”。缓存应有版本、计算窗口和最后成功时间。


## 19. 幂等、重试与并发

XXL-JOB失败重试会再次查询和写入。热点计算最好做到幂等：同一时间窗口重复执行，结果应相同或只发布新版本，不产生重复文章。

检查清单：

- 频道 key 使用覆盖写或版本发布，不用 `LPUSH` 造成重复文章；
- 结果由主数据和统计快照决定，不能随机排序；
- 两个节点同时运行时，最终缓存仍可读且版本可辨识；
- 失败重试不会删除上一批有效缓存；
- 超时旧实例继续写入时，批次版本能隔离新旧结果；
- JSON读取失败时回退并记录 key/版本。

`@Transactional`只能覆盖本地数据库事务，不能回滚 Feign 或 Redis。大批量处理可按频道/文章ID分片并记录批次状态；不要无限增加线程，避免数据库、Redis和远程服务同时被打满。


## 20. 监控与故障排查

建议监控：调度触发/失败/重试、执行器在线节点和耗时、数据库查询与连接池、Feign耗时/超时、Redis写入/命中率/内存、文章数/频道数/缓存批次和最后成功时间。

排查顺序：

1. 调度中心是否触发了正确任务和执行器；
2. 执行器是否注册在线，JobHandler名称是否完全匹配；
3. 任务日志是否进入业务方法；
4. 时间窗口和文章配置是否筛掉了数据；
5. Feign频道列表是否成功、响应码是否正确处理；
6. Redis key是否和首页 `dto.tag`一致，JSON是否可解析；
7. `/load`是否只在首屏读缓存，缓存缺失是否回退数据库。

“调度中心显示成功”不是最终业务成功标准，还要验证 Redis 批次、文章可见性和 APP响应。


## 21. 课程代码风险点与修正建议

| 讲义现象 | 风险 | 建议 |
|---|---|---|
| “前5天”与 `minusDays(50)`不一致 | 查询窗口扩大，压力和结果都变 | 统一需求、代码、注释、测试和监控 |
| 直接逐个 `set` | 半失败时出现新旧批次混合 | 批次版本+指针，或保留旧缓存 |
| Feign失败处理不清晰 | 频道缓存缺失或只更新默认推荐 | 区分空列表和远程失败，重试/告警 |
| `Integer score` | 计数大时溢出 | 使用更大类型或上限保护 |
| 只按score排序 | 同分顺序不稳定 | 增加发布时间、文章ID次排序 |
| 全量窗口文章加载内存 | 文章增长后OOM或长耗时 | 分页、分片、索引和批处理 |
| `LEFT JOIN`后过滤右表 | 配置缺失文章被排除 | 明确缺失语义，补约束或改查询 |
| 本地事务覆盖远程/Redis的错觉 | 半成功无法回滚 | 批次、补偿、幂等发布 |
| 缓存直接返回旧VO | 字段或上下架状态过期 | 统一响应模型并做可见性校验 |

这些是工程复习建议，不声称讲义代码已包含。


## 22. 测试与验收清单（本次未执行）

### XXL-JOB基础

1. 单执行器注册成功，JobHandler匹配；
2. 手动触发一次任务，调度和执行日志均成功；
3. Cron到点触发，参数传递正确；
4. 执行器宕机时故障转移或失败告警符合配置；
5. 执行器忙时阻塞策略符合预期；
6. 两节点轮询时同一次单节点任务只执行一次；
7. 分片广播时每个分片只处理自己的数据。

### 热点计算与首页

1. 有效文章按窗口筛选，下架/删除文章不进入缓存；
2. `views + likes×3 + comment×5 + collection×8`分值正确；
3. 各频道和默认推荐都不超过30条；
4. 同分排序稳定；
5. Feign、Redis、数据库失败能被调度中心识别为失败；
6. 重试不会重复列表或删除上一批缓存；
7. 首页首屏命中正确频道 key，加载更多走原分页；
8. 缓存不存在、JSON损坏或 Redis不可用时回退数据库；
9. 新文章、行为变化和下一次任务之间的延迟符合产品约定。

本篇只完成资料阅读、笔记生成与文件校验，没有启动课程环境或执行业务测试。


## 23. 复习主线与自测

~~~text
用户行为统计 → 定时计算score → 每频道Top30 → Redis快照
                                          ↓
APP首页首屏：先读热点；没有热点：回退原查询
~~~

- XXL-JOB由调度中心管理任务，执行器注册并执行 `@XxlJob` 方法。
- 轮询选择一个执行器；分片广播让所有节点执行并携带 shardIndex/shardTotal。
- 热点分值示例：阅读1、点赞3、评论5、收藏8；缓存每频道30条和默认30条。
- `hot_article_first_page_`是课程key前缀，`__all__`是默认标签约定。
- 讲义代码中“前5天”与 `minusDays(50)`冲突，必须先确认时间窗口。
- 本地事务不能回滚 Feign 和 Redis；任务需要幂等、批次和失败补偿。

### 自测题

1. **为什么按发布时间排序不能代表热点？** 新文章天然靠前，但没有经过行为积累；高访问量也会给数据库造成压力。
2. **轮询和分片广播的区别？** 轮询一次选择一个执行器；分片广播把一次调度广播到全部执行器并由分片参数分工。
3. **为什么任务重试必须幂等？** 非幂等写入会重复数据、重复计数或破坏缓存。
4. **为什么还要写默认推荐 key？** 用户可能查询全部频道，`__all__`提供不按频道过滤的Top30。
5. **为什么要记录5/50天冲突？** 时间窗口决定扫描压力和热点范围，不能凭代码或注释单方面猜测。
6. **缓存命中后为什么仍可能校验可见性？** 文章可能在上次定时计算后下架或删除，缓存是快照。
7. **Feign失败能当成空频道列表吗？** 不能；空列表和远程失败语义不同，应保留旧缓存或告警重试。
8. **为什么 `@Transactional`不能保证热点任务全成全败？** Feign和Redis在本地数据库事务之外，无法被同一事务回滚。

### 资料覆盖记录

| 已读取资料 | 本篇覆盖 |
|---|---|
| `热点文章-定时计算.md` | 需求分析、XXL-JOB、调度中心/执行器、路由/阻塞/分片、热点计算、Feign、Redis和首页改造 |

未读取代码、PPT、视频、资料目录和调度 SQL；部署、测试、监控、批次发布和可靠性方案均已标为工程补充。笔记完成不等于项目已经运行。
